# ResNet
**结构**：ResNet沿用了VGG完整的$3\times 3$卷积层设计。每个残差块包含两个$3\times 3$卷积层，以及一个跨层数据通路（shortcut/skip connection）。
**核心思想**：
1. **残差学习 (Residual Learning)**：让网络去学习残差映射 $f(\mathbf{x}) - \mathbf{x}$，而不是直接学习期望的底层映射。这使得网络即使非常深也能容易训练。
2. **跳跃连接 (Skip Connection)**：引入恒等映射（Identity Mapping）作为默认选项。如果某一层学不到有用特征，它可以“退化”成什么都不做（恒等映射），保证模型性能至少不会下降。
3. **解决梯度消失/爆炸**：通过直接将输入传到后面，梯度能更顺畅地反向流动。

![一个正常块（左图）和一个残差块（右图）。](../img/residual-block.svg)

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

class Residual(nn.Module):
    """ResNet 的残差块"""
    def __init__(self, input_channels, num_channels,
                 use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels,
                               kernel_size=3, padding=1, stride=strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels,
                               kernel_size=3, padding=1)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels,
                                   kernel_size=1, stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

def resnet_block(input_channels, num_channels, num_residuals,
                 first_block=False):
    """ResNet 的一个 Stage (包含多个 Residual block)"""
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(input_channels, num_channels,
                                use_1x1conv=True, strides=2))
        else:
            blk.append(Residual(num_channels, num_channels))
    return blk

# ResNet-18 主体架构
b1 = nn.Sequential(nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
                   nn.BatchNorm2d(64), nn.ReLU(),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block=True))
b3 = nn.Sequential(*resnet_block(64, 128, 2))
b4 = nn.Sequential(*resnet_block(128, 256, 2))
b5 = nn.Sequential(*resnet_block(256, 512, 2))

net = nn.Sequential(b1, b2, b3, b4, b5,
                    nn.AdaptiveAvgPool2d((1,1)),
                    nn.Flatten(), nn.Linear(512, 10))

# 检查各层输出形状
X = torch.rand(size=(1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t', X.shape)

# 训练
lr, num_epochs, batch_size = 0.05, 10, 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize=96)
d2l.train_ch6(net, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())

### 练习
1. **Inception 块与残差块的区别？**
    *   Inception 是并联不同尺度的卷积（宽）。
    *   ResNet 是卷积加上一条直连通路（深）。
2. **为什么有了嵌套函数类（nested function classes）还要限制复杂度？**
    *   虽然嵌套通过恒等映射保证了“不会变差”，但过高的复杂度会导致过拟合，以及计算资源的浪费。